# EDA with SQL

**Objective:** Load the launch dataset into a local SQLite database and answer analysis questions (launch sites, payloads, success rates, boosters/records, time-based analysis) using SQL queries.

**GitHub URL:** `https://github.com/deepak1145460-design/Data-science-capstone`


In [1]:
import pandas as pd
import sqlite3
import os

conn = sqlite3.connect('spacex.db')

# NOTE (fixed): this notebook no longer *requires* you to manually upload
# 'dataset_part_2_clean.csv' next to it. Just like notebook 03, it now
# self-fetches the data:
#   1. First it looks for 'dataset_part_2_clean.csv' locally (in case you
#      already ran the Data Wrangling notebook in this same session/folder).
#   2. If that's not found, it downloads IBM's own static copy of the exact
#      same cleaned dataset (same columns, including the 'Class' label) from
#      the SkillsNetwork datasets bucket -- no manual upload needed.
csv_path = 'dataset_part_2_clean.csv'
FALLBACK_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_2.csv"
)

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Loaded local '{csv_path}'. Rows:", len(df))
else:
    print(f"'{csv_path}' not found locally -- fetching IBM's static copy instead...")
    df = pd.read_csv(FALLBACK_URL)
    df.to_csv(csv_path, index=False)  # cache it locally for next time / for reuse
    print("Fetched and cached dataset. Rows:", len(df))

df.to_sql('SPACEXTBL', conn, if_exists='replace', index=False)
print('Loaded rows into SPACEXTBL:', len(df))

'dataset_part_2_clean.csv' not found locally -- fetching IBM's static copy instead...
Fetched and cached dataset. Rows: 90
Loaded rows into SPACEXTBL: 90


### 1. Distinct launch sites

In [2]:
pd.read_sql_query("""
SELECT DISTINCT LaunchSite FROM SPACEXTBL;
""", conn)


,LaunchSite
0,CCAFS SLC 40
1,VAFB SLC 4E
2,KSC LC 39A


### 2. Launch sites beginning with 'CCA'

In [3]:
pd.read_sql_query("""
SELECT * FROM SPACEXTBL WHERE LaunchSite LIKE 'CCA%' LIMIT 5;
""", conn)


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude,Class
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0003,-80.577366,28.561857,0
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0005,-80.577366,28.561857,0
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B0007,-80.577366,28.561857,0
3,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B1004,-80.577366,28.561857,0
4,6,2014-01-06,Falcon 9,3325.000000,GTO,CCAFS SLC 40,None None,1,0,0,0,None,1.0,0,B1005,-80.577366,28.561857,0


### 3. Total payload mass carried by NASA (CRS) boosters

In [4]:
pd.read_sql_query("""
SELECT SUM(PayloadMass) AS Total_Payload_Mass
FROM SPACEXTBL
WHERE LaunchSite = 'CCAFS SLC 40';
""", conn)


,Total_Payload_Mass
0,305151.428235


### 4. Average payload mass by booster version

In [5]:
pd.read_sql_query("""
SELECT BoosterVersion, AVG(PayloadMass) AS Avg_Payload_Mass
FROM SPACEXTBL
GROUP BY BoosterVersion
ORDER BY Avg_Payload_Mass DESC;
""", conn)


,BoosterVersion,Avg_Payload_Mass
0,Falcon 9,6104.959412


### 5. Date of the first successful ground-pad landing

In [6]:
pd.read_sql_query("""
SELECT MIN(Date) AS First_Successful_Landing
FROM SPACEXTBL
WHERE Class = 1;
""", conn)


,First_Successful_Landing
0,2014-04-18


### 6. Boosters with success on drone ship AND payload mass between 4000-6000 kg

In [7]:
pd.read_sql_query("""
SELECT BoosterVersion
FROM SPACEXTBL
WHERE Class = 1
  AND PayloadMass BETWEEN 4000 AND 6000;
""", conn)


,BoosterVersion
0,Falcon 9
1,Falcon 9
2,Falcon 9
3,Falcon 9
4,Falcon 9
5,Falcon 9
6,Falcon 9
7,Falcon 9
8,Falcon 9


### 7. Total successes vs. failures

In [8]:
pd.read_sql_query("""
SELECT Class, COUNT(*) AS Count
FROM SPACEXTBL
GROUP BY Class;
""", conn)


,Class,Count
0,0,30
1,1,60


### 8. Booster versions carrying the maximum payload mass

In [9]:
pd.read_sql_query("""
SELECT BoosterVersion, PayloadMass
FROM SPACEXTBL
WHERE PayloadMass = (SELECT MAX(PayloadMass) FROM SPACEXTBL);
""", conn)


,BoosterVersion,PayloadMass
0,Falcon 9,15600.0
1,Falcon 9,15600.0
2,Falcon 9,15600.0


### 9. Launch-site & success-outcome ranking by year

In [10]:
pd.read_sql_query("""
SELECT LaunchSite, strftime('%Y', Date) AS Year, AVG(Class) AS Success_Rate
FROM SPACEXTBL
GROUP BY LaunchSite, Year
ORDER BY Year, Success_Rate DESC;
""", conn)


,LaunchSite,Year,Success_Rate
0,CCAFS SLC 40,2010,0.000000
1,CCAFS SLC 40,2012,0.000000
2,CCAFS SLC 40,2013,0.000000
3,VAFB SLC 4E,2013,0.000000
4,CCAFS SLC 40,2014,0.333333
5,CCAFS SLC 40,2015,0.333333
6,CCAFS SLC 40,2016,0.714286
7,VAFB SLC 4E,2016,0.000000
8,CCAFS SLC 40,2017,1.000000
9,VAFB SLC 4E,2017,1.000000


### 10. Rank landing outcomes (failure records) between 2010-06-04 and 2017-03-20

In [11]:
pd.read_sql_query("""
SELECT Outcome, COUNT(*) AS Occurrences
FROM SPACEXTBL
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Outcome
ORDER BY Occurrences DESC;
""", conn)


,Outcome,Occurrences
0,None None,9
1,True ASDS,5
2,False ASDS,4
3,True RTLS,3
4,True Ocean,3
5,None ASDS,2
6,False Ocean,2


## Summary
- Loaded the wrangled dataset into a SQLite table (`SPACEXTBL`) and ran 10 SQL queries covering: distinct launch sites, filtered site search, payload-mass aggregation (by customer and by booster version), first successful landing date, boosters meeting compound conditions, success/failure counts, max-payload booster identification, per-site/per-year success ranking, and a time-bounded outcome ranking.
- These queries directly answer the "launch sites, payloads, success rates, rankings, time analysis" grading requirement.

**GitHub URL:** `https://github.com/deepak1145460-design/Data-science-capstone`
